# 🛰️ Vellakkal Methane Hotspot Detection — Live Inference
**GEE Project:** `methane-detection-research` | **Date:** 2026-08-26

### Scientific Honesty Notice
This notebook reports three distinct signals:
- **S5P CH₄ (ppb):** Raw TROPOMI column-averaged mixing ratio (~7×5.5 km footprint)
- **CH₄ anomaly (ppb):** Enhancement above 25 km local background; not an emission rate
- **CNN probability:** ResNet-50 prediction from Sentinel-2 spectral patch — correlates with surface conditions, not a direct CH₄ measurement
- **Hotspot label:** Derived classification from CNN prob ≥ 0.5; uncertainty not independently validated here

**No ground-truth labels exist for today → accuracy/F1 cannot be computed. Confidence is reported as CNN probability only.**

---
**Before running:** Runtime → Change Runtime Type → GPU (T4)

## Step 0: Install & Mount Drive

In [ ]:
!pip install -q earthengine-api==0.1.384 tensorflow geopandas shapely rasterio scipy folium tqdm
from google.colab import drive
drive.mount('/content/drive')
print('✅ Done')

## Step 1: Configuration — All constants in one place

In [ ]:
import os, sys, json, time, warnings
import numpy as np
import pandas as pd
from datetime import date, datetime, timedelta
warnings.filterwarnings('ignore')

# ── GEE ──────────────────────────────────────────────────────────
GEE_PROJECT_ID = 'methane-detection-research'

# ── Drive paths ──────────────────────────────────────────────────
# Adjust PROJECT_DRIVE_PATH if your Drive folder name differs
PROJECT_DRIVE_PATH = '/content/drive/MyDrive/methanedet_colab'
MODEL_DIR = os.path.join(PROJECT_DRIVE_PATH, 'models', 'ms_resnet50_2026-08-25')

# ── Target date ──────────────────────────────────────────────────
TARGET_DATE = '2026-08-26'   # change if re-running on a different day

# ── Site ─────────────────────────────────────────────────────────
SITE_LAT, SITE_LON = 9.8732, 78.0813   # Vellakkal Dumpsite centroid
SITE_NAME = 'Vellakkal_Madurai'

# ── Grid ─────────────────────────────────────────────────────────
GRID_STEP_DEG  = 0.003   # ≈ 333 m per step
GRID_RADIUS_DEG = 0.055  # ≈ 6 km radius → up to 37×37 = 1369 pts

# ── Sentinel-2 ───────────────────────────────────────────────────
S2_COLLECTION   = 'COPERNICUS/S2_SR_HARMONIZED'
BANDS_10M       = ['B2', 'B3', 'B4', 'B8']
BANDS_20M       = ['B11', 'B12']
# SCL cloud classes to mask: cloud shadow, med/high cloud prob, cirrus, snow
SCL_BAD_CLASSES = [3, 8, 9, 10, 11]
MAX_PATCH_CLOUD_FRAC = 0.10  # reject patch if >10% bad pixels
S2_WINDOW_DAYS  = 10         # ±10 days search (monsoon season)
DN_SCALE        = 10000.0
PIXEL_SIZE_M    = 10

# ── Sentinel-5P ──────────────────────────────────────────────────
CH4_BAND    = 'CH4_column_volume_mixing_ratio_dry_air'
S5P_QA_BAND = 'qa_value'
S5P_MIN_QA  = 0.5    # TROPOMI-recommended minimum QA threshold
S5P_WINDOW_DAYS = 7  # ±7 days for CH4 fetch; background uses 14 days
BG_RADIUS_M = 25000  # 25 km background ring
ENHANCEMENT_SIGMA_THRESHOLD = 1.5  # hotspot if enhancement > BG_STD × this

# ── Model (from preprocessing_config.json — DO NOT CHANGE) ───────
BAND_ORDER  = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI']
PATCH_SIZE  = 32
TRAIN_MEAN  = np.array([0.08401623, 0.10818147, 0.11733060,
                         0.36089358, 0.21295287, 0.17294672, 0.51829457], dtype='float32')
TRAIN_STD   = np.array([0.02606612, 0.02615055, 0.03849075,
                         0.08586697, 0.06056928, 0.06059426, 0.19674271], dtype='float32')

# ── Output paths ─────────────────────────────────────────────────
OUT_DIR_PRED = 'outputs/predictions'
OUT_DIR_GIS  = 'outputs/gis'
OUT_DIR_FIG  = 'outputs/figures'
for d in [OUT_DIR_PRED, OUT_DIR_GIS, OUT_DIR_FIG]:
    os.makedirs(d, exist_ok=True)

OUT_CSV_ALL     = f'{OUT_DIR_PRED}/vellakkal_all_predictions_{TARGET_DATE}.csv'
OUT_CSV_HOTSPOT = f'{OUT_DIR_PRED}/vellakkal_hotspot_zones_{TARGET_DATE}.csv'
OUT_GEOJSON     = f'{OUT_DIR_GIS}/vellakkal_hotspots_{TARGET_DATE}.geojson'
OUT_TIF         = f'{OUT_DIR_GIS}/vellakkal_probability_surface_{TARGET_DATE}.tif'
OUT_MAP         = f'{OUT_DIR_GIS}/vellakkal_live_map_{TARGET_DATE}.html'

sys.path.insert(0, PROJECT_DRIVE_PATH)

print('Configuration loaded.')
print(f'  GEE project  : {GEE_PROJECT_ID}')
print(f'  Target date  : {TARGET_DATE}')
print(f'  Model dir    : {MODEL_DIR}')
print(f'  Band order   : {BAND_ORDER}')
print(f'  Patch size   : {PATCH_SIZE}×{PATCH_SIZE} @ {PIXEL_SIZE_M}m = {PATCH_SIZE*PIXEL_SIZE_M}m')
print(f'  TRAIN_MEAN   : {TRAIN_MEAN}')
print(f'  TRAIN_STD    : {TRAIN_STD}')

## Step 2: GEE Authentication

In [ ]:
import ee
try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print('✅ GEE already authenticated.')
except Exception:
    print('Opening browser auth...')
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
    print('✅ GEE authenticated.')

# Sanity test
assert ee.Number(1).getInfo() == 1, 'GEE connection failed'
print(f'GEE OK | project = {GEE_PROJECT_ID}')

## Step 3: Check Satellite Availability & Select Best Collections

In [ ]:
site_pt = ee.Geometry.Point([SITE_LON, SITE_LAT])
tgt     = ee.Date(TARGET_DATE)
w_start = tgt.advance(-S2_WINDOW_DAYS, 'day').format('YYYY-MM-dd').getInfo()
w_end   = tgt.advance(1, 'day').format('YYYY-MM-dd').getInfo()

print('=' * 65)
print(f'  Satellite availability check  [{w_start} → {w_end}]')
print('=' * 65)

# ── Sentinel-5P ──
nrti_n = (ee.ImageCollection('COPERNICUS/S5P/NRTI/L3_CH4')
          .filterBounds(site_pt).filterDate(w_start, w_end).size().getInfo())
offl_n = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_CH4')
          .filterBounds(site_pt).filterDate(w_start, w_end).size().getInfo())

print(f'  S5P OFFL (5-day delay, higher QA): {offl_n} granules')
print(f'  S5P NRTI (3-hour delay, lower QA): {nrti_n} granules')

if offl_count := offl_n:
    S5P_COLLECTION = 'COPERNICUS/S5P/OFFL/L3_CH4'
    s5p_label = 'OFFL'
    print(f'  → Using OFFL (preferred quality)')
elif nrti_count := nrti_n:
    S5P_COLLECTION = 'COPERNICUS/S5P/NRTI/L3_CH4'
    s5p_label = 'NRTI'
    print(f'  → OFFL not yet available. Using NRTI.')
else:
    # Fall back: find the most recent OFFL data
    print('  → No S5P data in ±10 day window. Finding most recent available...')
    S5P_COLLECTION = 'COPERNICUS/S5P/OFFL/L3_CH4'
    s5p_label = 'OFFL-fallback'
    recent = (ee.ImageCollection(S5P_COLLECTION)
              .filterBounds(site_pt)
              .sort('system:time_start', False)
              .first())
    recent_date = ee.Date(recent.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    print(f'  → Most recent S5P OFFL: {recent_date}  ⚠️ Data is NOT from {TARGET_DATE}')

# ── Sentinel-2 ──
s2_coll = (ee.ImageCollection(S2_COLLECTION)
           .filterBounds(site_pt)
           .filterDate(w_start, w_end)
           .sort('CLOUDY_PIXEL_PERCENTAGE'))
s2_n    = s2_coll.size().getInfo()
print(f'\n  Sentinel-2 SR scenes in window : {s2_n}')

if s2_n > 0:
    best_s2    = s2_coll.first()
    best_cloud = best_s2.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
    best_date  = ee.Date(best_s2.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    best_id    = best_s2.get('system:index').getInfo()
    print(f'  Best scene : {best_date}  cloud={best_cloud:.1f}%  id={best_id}')
    if best_cloud > 70:
        print('  ⚠️  High scene-level cloud cover (monsoon). Per-patch cloud masking will filter.')
    if best_date != TARGET_DATE:
        print(f'  ℹ️  Best scene is from {best_date}, not {TARGET_DATE}. This will be reported in outputs.')
else:
    raise RuntimeError('No Sentinel-2 scenes found. Widen S2_WINDOW_DAYS or check GEE quota.')

print('=' * 65)

## Step 4: Compute CH₄ Background (25 km ring, ±14 days)

In [ ]:
bg_region = site_pt.buffer(BG_RADIUS_M)
bg_start  = tgt.advance(-14, 'day')
bg_end    = tgt.advance(1, 'day')

bg_coll = (
    ee.ImageCollection(S5P_COLLECTION)
    .filterBounds(bg_region)
    .filterDate(bg_start, bg_end)
    .map(lambda img: img.updateMask(img.select(S5P_QA_BAND).gte(S5P_MIN_QA)))
    .select(CH4_BAND)
)
bg_n = bg_coll.size().getInfo()
print(f'Background S5P images (25 km ring, ±14 days): {bg_n}')

if bg_n > 0:
    stats = bg_coll.reduce(
        ee.Reducer.median().combine(ee.Reducer.stdDev(), sharedInputs=True)
    ).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=bg_region, scale=7000, maxPixels=1e7
    )
    BG_MEDIAN = stats.get(f'{CH4_BAND}_median').getInfo()
    BG_STD    = stats.get(f'{CH4_BAND}_stdDev').getInfo()
    bg_source = 'GEE'
else:
    # Climatological fallback (Madurai region, August)
    BG_MEDIAN = 1912.5
    BG_STD    = 9.2
    bg_source = 'climatological-fallback'
    print('⚠️  No S5P data for background — using climatological fallback values.')

if BG_MEDIAN is None:
    BG_MEDIAN = 1912.5
    BG_STD    = 9.2
    bg_source = 'climatological-fallback'
    print('⚠️  Background stats returned None — using fallback.')

HOTSPOT_CH4_THRESH = BG_MEDIAN + ENHANCEMENT_SIGMA_THRESHOLD * BG_STD

print(f'\nCH₄ background ({bg_source}):')
print(f'  Median  : {BG_MEDIAN:.3f} ppb')
print(f'  Std Dev : {BG_STD:.3f} ppb')
print(f'  Hotspot threshold (median + {ENHANCEMENT_SIGMA_THRESHOLD}σ): {HOTSPOT_CH4_THRESH:.3f} ppb')
print(f'  Units   : ppb (dry-air CH₄ column mixing ratio, TROPOMI L2)')
print()
print('Note: TROPOMI CH₄ has ~7×5.5 km footprint. Enhancement ≠ emission source location.')

## Step 5: Load Trained Model (exact config from preprocessing_config.json)

In [ ]:
import tensorflow as tf
from tensorflow import keras

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

# --- Verify preprocessing_config.json matches hardcoded constants ---
cfg_path = os.path.join(MODEL_DIR, 'preprocessing_config.json')
with open(cfg_path) as f:
    cfg = json.load(f)

assert cfg['band_order'] == BAND_ORDER, f"Band order mismatch!\nConfig: {cfg['band_order']}\nHardcoded: {BAND_ORDER}"
assert cfg['patch_size'] == PATCH_SIZE, f"Patch size mismatch: {cfg['patch_size']} vs {PATCH_SIZE}"
np.testing.assert_allclose(cfg['mean'], TRAIN_MEAN, rtol=1e-5,
    err_msg='TRAIN_MEAN mismatch between config and hardcoded constant')
np.testing.assert_allclose(cfg['std'],  TRAIN_STD,  rtol=1e-5,
    err_msg='TRAIN_STD mismatch between config and hardcoded constant')
print('✅ preprocessing_config.json matches hardcoded constants exactly.')

# --- Load model with custom layers ---
from model import SqueezeAndExcitationBlock, SpatialAttentionBlock

model_path = os.path.join(MODEL_DIR, 'model.keras')
print(f'Loading: {model_path}')
model = keras.models.load_model(
    model_path,
    custom_objects={
        'SqueezeAndExcitationBlock': SqueezeAndExcitationBlock,
        'SpatialAttentionBlock':     SpatialAttentionBlock,
    }
)
print(f'✅ Model loaded: {model.name}')
print(f'   Input shape  : {model.input_shape}')
print(f'   Output keys  : {list(model.output.keys()) if isinstance(model.output, dict) else "single"}')

# --- Verify model accepts expected input shape ---
test_input = np.zeros((1, PATCH_SIZE, PATCH_SIZE, len(BAND_ORDER)), dtype='float32')
test_out   = model.predict(test_input, verbose=0)
if isinstance(test_out, dict):
    MODEL_MULTIHEAD = True
    print(f'   Output type  : multi-head {list(test_out.keys())}')
    print(f'   hotspot_prob shape: {test_out["hotspot_prob"].shape}')
else:
    MODEL_MULTIHEAD = False
    print(f'   Output type  : single-head, shape {test_out.shape}')
print('✅ Warmup OK — model ready.')

## Step 6: GEE Helper Functions (Sentinel-2 patch + S5P CH₄)

In [ ]:
def build_s2_patch_image(scene):
    """Build 7-channel image matching training pipeline exactly: B2,B3,B4,B8,B11,B12,NDVI."""
    b10  = scene.select(BANDS_10M)
    b20  = scene.select(BANDS_20M).resample('bilinear').reproject(
               crs=scene.select('B2').projection(), scale=PIXEL_SIZE_M)
    ndvi = scene.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return b10.addBands(b20).addBands(ndvi)


def compute_cloud_frac(scene, region):
    """Fraction of pixels in region that are cloud/shadow/snow per SCL."""
    scl     = scene.select('SCL')
    bad     = scl.remap(SCL_BAD_CLASSES, [1] * len(SCL_BAD_CLASSES), 0).rename('bad')
    stats   = bad.reduceRegion(ee.Reducer.mean(), region, PIXEL_SIZE_M, maxPixels=1e6)
    val     = stats.get('bad').getInfo()
    return float(val) if val is not None else 1.0


def fetch_s2_patch(lon, lat):
    """
    Fetch one Sentinel-2 patch at (lon, lat) centered on TARGET_DATE ± S2_WINDOW_DAYS.
    Returns (patch_array [32,32,7] float32 in reflectance, metadata_dict)
    or (None, metadata_dict) if no valid cloud-free patch exists.
    Band order: B2, B3, B4, B8, B11, B12, NDVI  (matches BAND_ORDER)
    """
    point  = ee.Geometry.Point([lon, lat])
    region = point.buffer(PATCH_SIZE / 2 * PIXEL_SIZE_M).bounds()
    start  = tgt.advance(-S2_WINDOW_DAYS, 'day')
    end    = tgt.advance(S2_WINDOW_DAYS, 'day')

    coll = (
        ee.ImageCollection(S2_COLLECTION)
        .filterBounds(point)
        .filterDate(start, end)
        .sort('CLOUDY_PIXEL_PERCENTAGE')
    )
    if coll.size().getInfo() == 0:
        return None, {'status': 'no_s2_scene', 'scene_id': None,
                      's2_date': None, 'scene_cloud_pct': None, 'patch_cloud_frac': None}

    scene      = coll.first()
    scene_id   = scene.get('system:index').getInfo()
    s2_date    = ee.Date(scene.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    scene_cld  = scene.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()

    patch_cld  = compute_cloud_frac(scene, region)
    meta       = {'scene_id': scene_id, 's2_date': s2_date,
                  'scene_cloud_pct': round(scene_cld, 2),
                  'patch_cloud_frac': round(patch_cld, 3)}

    if patch_cld > MAX_PATCH_CLOUD_FRAC:
        meta['status'] = 'rejected_cloudy'
        return None, meta

    # Sample pixel values
    patch_img = build_s2_patch_image(scene)
    sampled   = patch_img.sampleRectangle(region=region, defaultValue=0)

    arrays = []
    for band in BAND_ORDER:  # strict order from config
        arr = np.array(sampled.get(band).getInfo(), dtype='float32')
        out = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype='float32')
        h, w = arr.shape
        ch, cw = min(h, PATCH_SIZE), min(w, PATCH_SIZE)
        out[:ch, :cw] = arr[:ch, :cw]
        arrays.append(out)

    patch = np.stack(arrays, axis=-1)  # (32, 32, 7)
    # DN → reflectance for reflectance bands; NDVI is already [-1,1]
    patch[..., :6] /= DN_SCALE

    # Validate: reflectance bands should be in [0, 1]
    if patch[..., :6].max() > 1.5:
        meta['status'] = 'error_dn_scale'
        return None, meta

    meta['status'] = 'ok'
    return patch, meta


def fetch_ch4_value(lon, lat):
    """
    Fetch closest-in-time, QA-passing S5P CH4 at (lon, lat) ± S5P_WINDOW_DAYS.
    Returns (ch4_ppb, ch4_qa, s5p_date, s5p_granule_id) or Nones.
    """
    point  = ee.Geometry.Point([lon, lat])
    start  = tgt.advance(-S5P_WINDOW_DAYS, 'day')
    end    = tgt.advance(1, 'day')

    coll = (
        ee.ImageCollection(S5P_COLLECTION)
        .filterBounds(point)
        .filterDate(start, end)
        .map(lambda img: img.set(
            'time_diff',
            ee.Number(img.get('system:time_start')).subtract(tgt.millis()).abs()
        ))
        .sort('time_diff')
    )
    if coll.size().getInfo() == 0:
        return None, None, None, None

    img        = coll.first()
    granule_id = img.get('system:index').getInfo()
    s5p_date   = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd').getInfo()

    qa_val = img.select(S5P_QA_BAND).reduceRegion(
        ee.Reducer.first(), point, 7000, maxPixels=1e6).get(S5P_QA_BAND).getInfo()

    if qa_val is None or float(qa_val) < S5P_MIN_QA:
        return None, qa_val, s5p_date, granule_id  # QA fail

    ch4_val = img.select(CH4_BAND).reduceRegion(
        ee.Reducer.first(), point, 7000, maxPixels=1e6).get(CH4_BAND).getInfo()

    return ch4_val, qa_val, s5p_date, granule_id


def normalize_patch(patch_reflectance):
    """Apply training-only normalization. Input: (32,32,7) reflectance. Output: normalized."""
    return (patch_reflectance - TRAIN_MEAN) / TRAIN_STD


def predict_patch(patch_norm):
    """Run model on a single normalized patch. Returns (prob, enhancement_ppb_or_None)."""
    out = model.predict(patch_norm[None, ...], verbose=0)
    if MODEL_MULTIHEAD:
        return float(out['hotspot_prob'][0, 0]), float(out['enhancement_ppb'][0, 0])
    else:
        return float(out[0, 0]), None


print('✅ GEE helper functions defined.')

## Step 7: Test Inference on 3 Points (verify before full run)
Runs exactly 3 points to confirm the pipeline works end-to-end before spending ~6 min on full grid.

In [ ]:
test_points = [
    {'lat': SITE_LAT,           'lon': SITE_LON,           'label': 'dumpsite-centroid'},
    {'lat': SITE_LAT + 0.010,   'lon': SITE_LON + 0.010,   'label': 'near-1km-NE'},
    {'lat': SITE_LAT - 0.030,   'lon': SITE_LON - 0.030,   'label': 'outer-ring-SW'},
]

print('=' * 70)
print('  3-POINT TEST INFERENCE')
print('=' * 70)

for tp in test_points:
    lat, lon = tp['lat'], tp['lon']
    print(f"\n[{tp['label']}]  lat={lat:.4f}  lon={lon:.4f}")

    patch, s2m = fetch_s2_patch(lon, lat)
    print(f"  S2 status       : {s2m['status']}")
    print(f"  S2 date         : {s2m['s2_date']}")
    print(f"  Patch cloud frac: {s2m['patch_cloud_frac']}")

    if patch is not None:
        print(f"  Patch shape     : {patch.shape}")
        print(f"  B2 range        : [{patch[...,0].min():.4f}, {patch[...,0].max():.4f}]  (expected ~0.04-0.25)")
        print(f"  NDVI range      : [{patch[...,6].min():.4f}, {patch[...,6].max():.4f}]  (expected ~-1 to 1)")

        patch_norm = normalize_patch(patch)
        prob, enh_pred = predict_patch(patch_norm)
        print(f"  CNN prob        : {prob:.6f}  → {'HOTSPOT' if prob >= 0.5 else 'normal'}")
        if enh_pred is not None:
            print(f"  Predicted enh   : {enh_pred:.2f} ppb  (CNN estimate, not measured)")
    else:
        print('  Patch skipped — no prediction.')

    ch4, qa, s5p_date, gid = fetch_ch4_value(lon, lat)
    print(f"  S5P date        : {s5p_date}")
    print(f"  S5P QA          : {qa}  (min required: {S5P_MIN_QA})")
    print(f"  CH4 (ppb)       : {ch4}")
    if ch4 is not None:
        enh = ch4 - BG_MEDIAN
        print(f"  Enhancement     : {enh:+.2f} ppb  (above background {BG_MEDIAN:.2f} ppb)")
        print(f"  S5P hotspot     : {'YES' if ch4 > HOTSPOT_CH4_THRESH else 'no'}  (thresh={HOTSPOT_CH4_THRESH:.2f} ppb)")

print('\n✅ Test complete. If all looks reasonable, proceed to full run.')
print('⚠️  STOP HERE if S2 date is wrong or values are clearly out-of-range.')

## Step 8: Build Full 37×37 Grid & Run Batch Inference

Strategy to minimise GEE calls:
1. Group all points by the closest Sentinel-2 scene (same scene → same band sampling call structure)
2. S5P CH₄ is fetched per-point but cached by rounded lat/lon at 0.07° resolution (≈ S5P pixel size)
3. Batch predict all valid patches in one `model.predict()` call

In [ ]:
from tqdm.notebook import tqdm

# ── Build grid ────────────────────────────────────────────────────
lat_vals = np.arange(SITE_LAT - GRID_RADIUS_DEG,
                     SITE_LAT + GRID_RADIUS_DEG + GRID_STEP_DEG,
                     GRID_STEP_DEG)
lon_vals = np.arange(SITE_LON - GRID_RADIUS_DEG,
                     SITE_LON + GRID_RADIUS_DEG + GRID_STEP_DEG,
                     GRID_STEP_DEG)
grid_lats, grid_lons = np.meshgrid(lat_vals, lon_vals)
grid_lats = grid_lats.ravel()
grid_lons = grid_lons.ravel()

print(f'Grid: {len(grid_lats)} points  ({len(lat_vals)}×{len(lon_vals)})')
print(f'Coverage: ~{2*GRID_RADIUS_DEG*111:.1f} km × {2*GRID_RADIUS_DEG*111:.1f} km')

# ── S5P cache keyed at 0.07° resolution (~7 km, matches TROPOMI pixel) ──
s5p_cache = {}   # key: (round(lat,2), round(lon,2)) → (ch4, qa, s5p_date, gid)

def get_ch4_cached(lat, lon):
    key = (round(lat, 2), round(lon, 2))
    if key not in s5p_cache:
        s5p_cache[key] = fetch_ch4_value(lon, lat)
    return s5p_cache[key]

# ── Per-point fetch loop ──────────────────────────────────────────
records    = []   # raw records for all points
patches    = []   # valid normalized patches for batch inference
patch_idx  = []   # index into records[] for each patch

n_ok = n_cloudy = n_no_scene = n_err = 0

for i, (lat, lon) in enumerate(tqdm(zip(grid_lats, grid_lons),
                                     total=len(grid_lats), desc='Fetching patches')):
    # ── S2 patch ──
    patch, s2m = fetch_s2_patch(lon, lat)

    # ── S5P CH4 (cached) ──
    ch4, qa, s5p_date, gid = get_ch4_cached(lat, lon)
    ch4_enh   = (ch4 - BG_MEDIAN)                          if ch4 is not None else None
    ch4_s5p_hs = int(ch4 > HOTSPOT_CH4_THRESH)             if ch4 is not None else None

    rec = dict(
        lat=lat, lon=lon,
        # S2 metadata
        s2_scene_id=s2m.get('scene_id'),
        s2_acquisition_date=s2m.get('s2_date'),
        s2_scene_cloud_pct=s2m.get('scene_cloud_pct'),
        s2_patch_cloud_frac=s2m.get('patch_cloud_frac'),
        s2_status=s2m['status'],
        # S5P metadata
        s5p_granule_id=gid,
        s5p_acquisition_date=s5p_date,
        s5p_collection=S5P_COLLECTION.split('/')[-2],
        s5p_qa_value=qa,
        # CH4 values
        S5P_CH4_ppb=ch4,
        units='ppb (dry-air column mixing ratio)',
        ch4_background_ppb=round(BG_MEDIAN, 3),
        ch4_background_std_ppb=round(BG_STD, 3),
        ch4_anomaly_ppb=round(ch4_enh, 3) if ch4_enh is not None else None,
        s5p_hotspot_label=ch4_s5p_hs,
        s5p_hotspot_threshold_ppb=round(HOTSPOT_CH4_THRESH, 3),
        # CNN fields — filled after batch predict
        CNN_probability=None,
        predicted_class=None,
        CNN_predicted_enhancement_ppb=None,
        confidence='CNN probability (no independent ground truth for validation)',
    )

    if patch is not None and s2m['status'] == 'ok':
        patch_norm = normalize_patch(patch)
        patches.append(patch_norm)
        patch_idx.append(len(records))
        n_ok += 1
    elif s2m['status'] == 'rejected_cloudy':
        n_cloudy += 1
    elif s2m['status'] == 'no_s2_scene':
        n_no_scene += 1
    else:
        n_err += 1

    records.append(rec)
    time.sleep(0.04)  # gentle GEE rate-limit

print(f'\nFetch complete:')
print(f'  Patches ready for inference : {n_ok}')
print(f'  Rejected (cloudy)           : {n_cloudy}')
print(f'  No S2 scene                 : {n_no_scene}')
print(f'  Other errors                : {n_err}')
print(f'  S5P cache hits              : {sum(1 for v in s5p_cache.values() if v[0] is not None)} / {len(s5p_cache)} unique cells')

In [ ]:
# ── Batch inference ───────────────────────────────────────────────
print(f'Running batch inference on {len(patches)} patches...')

if len(patches) == 0:
    print('❌ No valid patches. All points were cloudy or missing. Try expanding S2_WINDOW_DAYS.')
else:
    batch_arr  = np.stack(patches, axis=0)   # (N, 32, 32, 7)
    batch_out  = model.predict(batch_arr, batch_size=32, verbose=1)

    if MODEL_MULTIHEAD:
        probs      = batch_out['hotspot_prob'].ravel()
        enh_preds  = batch_out['enhancement_ppb'].ravel()
    else:
        probs      = batch_out.ravel()
        enh_preds  = [None] * len(probs)

    # Write CNN results back into records
    for j, rec_i in enumerate(patch_idx):
        p = float(probs[j])
        records[rec_i]['CNN_probability']              = round(p, 6)
        records[rec_i]['predicted_class']              = int(p >= 0.5)
        if enh_preds[j] is not None:
            records[rec_i]['CNN_predicted_enhancement_ppb'] = round(float(enh_preds[j]), 3)

    n_hotspot  = sum(1 for r in records if r.get('predicted_class') == 1)
    print(f'\n✅ Batch inference done.')
    print(f'  Total points    : {len(records)}')
    print(f'  Scored          : {len(patches)}')
    print(f'  CNN hotspot=1   : {n_hotspot}  ({n_hotspot/len(patches)*100:.1f}% of scored)')
    print(f'  CNN mean prob   : {probs.mean():.4f}')
    print(f'  CNN max prob    : {probs.max():.4f}')
    print()
    print('NOTE: CNN probability reflects Sentinel-2 spectral surface patterns correlated')
    print('      with historical CH4 hotspot labels. It is NOT a direct CH4 measurement.')

## Step 9: Save All Required Output Files

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import rasterio
from rasterio.transform import from_origin
from scipy.spatial import cKDTree

results_df = pd.DataFrame(records)

# ── A: Full predictions CSV ─────────────────────────────────────
results_df.to_csv(OUT_CSV_ALL, index=False)
print(f'✅ All predictions  : {OUT_CSV_ALL}  ({len(results_df)} rows)')

# ── B: Hotspot-only CSV ─────────────────────────────────────────
hotspot_df = (
    results_df[results_df['predicted_class'] == 1]
    .sort_values('CNN_probability', ascending=False)
    .reset_index(drop=True)
)
hotspot_df.to_csv(OUT_CSV_HOTSPOT, index=False)
print(f'✅ Hotspot zones    : {OUT_CSV_HOTSPOT}  ({len(hotspot_df)} hotspots)')

# ── C: GeoJSON (QGIS/Google Earth compatible) ───────────────────
valid_df = results_df.dropna(subset=['CNN_probability'])
gdf = gpd.GeoDataFrame(
    valid_df,
    geometry=gpd.points_from_xy(valid_df['lon'], valid_df['lat']),
    crs='EPSG:4326'
)
gdf.to_file(OUT_GEOJSON, driver='GeoJSON')
print(f'✅ GeoJSON          : {OUT_GEOJSON}  ({len(gdf)} points)')

# ── D: IDW-Interpolated GeoTIFF ─────────────────────────────────
# Only points with valid CNN probability are interpolated.
# This raster is clearly an IDW interpolation of sparse point predictions,
# NOT a raw satellite product.
vals_v  = valid_df['CNN_probability'].values
lats_v  = valid_df['lat'].values
lons_v  = valid_df['lon'].values

RASTER_RES = 0.0015  # ~167 m (finer than grid for smooth visual, same order as S2 patch)
pad        = RASTER_RES * 5
lat_rng = np.arange(lats_v.max() + pad, lats_v.min() - pad, -RASTER_RES)
lon_rng = np.arange(lons_v.min() - pad, lons_v.max() + pad,  RASTER_RES)
glon, glat = np.meshgrid(lon_rng, lat_rng)

tree = cKDTree(np.column_stack([lons_v, lats_v]))
d, ix = tree.query(np.column_stack([glon.ravel(), glat.ravel()]),
                   k=min(8, len(vals_v)))
d = np.maximum(d, 1e-10)
if d.ndim == 1:
    d, ix = d[:, None], ix[:, None]
w       = 1.0 / d ** 2
surface = (np.sum(w * vals_v[ix], axis=1) / np.sum(w, axis=1)
           ).reshape(glat.shape).astype('float32')

transform = from_origin(
    lons_v.min() - pad, lats_v.max() + pad, RASTER_RES, RASTER_RES)

with rasterio.open(
    OUT_TIF, 'w', driver='GTiff',
    height=surface.shape[0], width=surface.shape[1],
    count=1, dtype='float32', crs='EPSG:4326', transform=transform,
    compress='lzw'
) as dst:
    dst.write(surface, 1)
    dst.update_tags(
        DESCRIPTION='IDW interpolation of CNN hotspot probabilities from sparse grid points',
        INTERPOLATION='Inverse Distance Weighting, k=8, power=2',
        SOURCE_SENSOR='Sentinel-2 SR + Sentinel-5P TROPOMI',
        MODEL='MultiSpectral_ResNet50 (ms_resnet50_2026-08-25)',
        TARGET_DATE=TARGET_DATE,
        S5P_COLLECTION=S5P_COLLECTION,
        BACKGROUND_PPB=str(round(BG_MEDIAN, 3)),
        WARNING='Interpolated visualization only — not a raw satellite measurement'
    )

print(f'✅ IDW GeoTIFF      : {OUT_TIF}  ({surface.shape[0]}×{surface.shape[1]} px @ {RASTER_RES}°)')
print(f'   → Coordinate reference system: EPSG:4326')
print(f'   → WARNING tag embedded: this is IDW interpolation, not a raw sensor product.')

print(f'\nTop 10 hotspot locations (by CNN probability):')
cols = ['lat','lon','CNN_probability','predicted_class',
        'S5P_CH4_ppb','ch4_anomaly_ppb','s5p_hotspot_label',
        's2_acquisition_date','s5p_acquisition_date',
        's2_patch_cloud_frac','s5p_qa_value']
display(hotspot_df[cols].head(10).round(4))

## Step 10: Interactive Folium Map

In [ ]:
import folium
from folium.plugins import HeatMap, MarkerCluster

m = folium.Map(
    location=[SITE_LAT, SITE_LON],
    zoom_start=14,
    tiles='CartoDB dark_matter'
)

# Title / legend note
title_html = f'''
<div style="position:fixed;top:10px;left:60px;z-index:9999;
     background:rgba(13,17,23,0.88);color:#ccc;
     padding:10px 14px;border-radius:8px;font-family:monospace;font-size:12px;">
<b>🛰️ Vellakkal Methane Hotspot — {TARGET_DATE}</b><br>
CNN probability (Sentinel-2 spectral classification)<br>
S5P CH₄ footprint ≈ 7×5.5 km — cannot pinpoint emission sources<br>
No ground truth → accuracy not validated<br>
<span style="color:#ff8c00">🟠 prob≥0.80</span> &nbsp;
<span style="color:#ffd700">🟡 prob≥0.65</span> &nbsp;
<span style="color:#90ee90">🟢 prob≥0.50</span>
</div>'''
m.get_root().html.add_child(folium.Element(title_html))

# ── Heatmap layer ──
heat_data = [
    [r['lat'], r['lon'], r['CNN_probability']]
    for r in records if r.get('CNN_probability') is not None
]
HeatMap(
    heat_data, name='CNN Hotspot Probability (Heatmap)',
    radius=20, blur=22,
    gradient={0.0: '#0000ff', 0.35: '#00cc00',
              0.60: '#ffff00', 0.80: '#ff8800', 1.0: '#ff0000'}
).add_to(m)

# ── Hotspot point markers ──
cluster = MarkerCluster(name='CNN Hotspot Points (prob ≥ 0.5)').add_to(m)
for r in hotspot_df.itertuples():
    p = r.CNN_probability
    color = 'red' if p >= 0.80 else 'orange' if p >= 0.65 else 'green'
    ch4_str  = f"{r.S5P_CH4_ppb:.2f} ppb" if pd.notna(r.S5P_CH4_ppb) else 'No valid S5P'
    enh_str  = f"{r.ch4_anomaly_ppb:+.2f} ppb" if pd.notna(r.ch4_anomaly_ppb) else 'N/A'
    s5p_hs   = 'YES' if r.s5p_hotspot_label == 1 else ('NO' if r.s5p_hotspot_label == 0 else 'no data')

    popup_html = (
        f"<b>CNN Hotspot</b><br>"
        f"Lat: {r.lat:.6f}°N<br>Lon: {r.lon:.6f}°E<br>"
        f"<b>CNN probability:</b> {p:.4f}<br>"
        f"<b>S5P CH₄:</b> {ch4_str}<br>"
        f"<b>CH₄ anomaly:</b> {enh_str}<br>"
        f"<b>S5P hotspot:</b> {s5p_hs}<br>"
        f"<b>S2 date:</b> {r.s2_acquisition_date}<br>"
        f"<b>S5P date:</b> {r.s5p_acquisition_date}<br>"
        f"<b>Patch cloud:</b> {r.s2_patch_cloud_frac}<br>"
        f"<small>Confidence: CNN prob only — no ground truth</small>"
    )
    folium.CircleMarker(
        location=[r.lat, r.lon],
        radius=7, color=color, fill=True,
        fill_color=color, fill_opacity=0.75, weight=1,
        popup=folium.Popup(popup_html, max_width=260),
        tooltip=f'CNN: {p:.3f}'
    ).add_to(cluster)

# ── Dumpsite & background ring markers ──
folium.Marker(
    [SITE_LAT, SITE_LON],
    popup=folium.Popup(
        f'<b>Vellakkal Dumpsite</b><br>Madurai, TN<br>'
        f'{SITE_LAT}°N, {SITE_LON}°E<br>'
        f'CH₄ background: {BG_MEDIAN:.2f} ppb<br>Threshold: {HOTSPOT_CH4_THRESH:.2f} ppb',
        max_width=220
    ),
    icon=folium.Icon(color='white', icon='industry', prefix='fa'),
    tooltip='Vellakkal Dumpsite'
).add_to(m)

folium.Circle([SITE_LAT, SITE_LON], radius=6000,
              color='#4488ff', weight=1.2, fill=False,
              tooltip='6 km study radius').add_to(m)
folium.Circle([SITE_LAT, SITE_LON], radius=25000,
              color='#aaaaaa', weight=0.8, fill=False, dash_array='6',
              tooltip='25 km CH₄ background ring').add_to(m)

folium.LayerControl().add_to(m)

m.save(OUT_MAP)
print(f'✅ Interactive map: {OUT_MAP}')
m

## Step 11: Summary Report (Scientific)

In [ ]:
import matplotlib.pyplot as plt

scored    = results_df.dropna(subset=['CNN_probability'])
n_scored  = len(scored)
n_hs_cnn  = (scored['predicted_class'] == 1).sum()
n_hs_s5p  = scored['s5p_hotspot_label'].eq(1).sum()
n_both    = ((scored['predicted_class'] == 1) & (scored['s5p_hotspot_label'] == 1)).sum()

print('═' * 70)
print(f'  VELLAKKAL METHANE HOTSPOT DETECTION REPORT')
print(f'  Date target      : {TARGET_DATE}')
print(f'  Run at (UTC+5:30): {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  GEE project      : {GEE_PROJECT_ID}')
print('─' * 70)
print(f'  Sentinel-2  : {S2_COLLECTION} (10 m, ±{S2_WINDOW_DAYS} days)')
print(f'  Sentinel-5P : {S5P_COLLECTION} (~7 km, ±{S5P_WINDOW_DAYS} days)')
print(f'  Model       : ms_resnet50_2026-08-25 ({cfg["backbone"]})')
print(f'  Input       : 32×32×7 patches {BAND_ORDER}')
print('─' * 70)
print(f'  CH₄ background   : {BG_MEDIAN:.3f} ppb  (25 km ring, ±14 days)')
print(f'  CH₄ std          : {BG_STD:.3f} ppb')
print(f'  Hotspot threshold: {HOTSPOT_CH4_THRESH:.3f} ppb  (bg + {ENHANCEMENT_SIGMA_THRESHOLD}σ)')
print(f'  Units            : ppb (dry-air column mixing ratio, L2 TROPOMI)')
print('─' * 70)
print(f'  Grid points      : {len(results_df)}')
print(f'  Scored (cloud OK): {n_scored}')
print(f'  Cloudy/rejected  : {n_cloudy}')
print(f'  No S2 scene      : {n_no_scene}')
print('─' * 70)
print(f'  CNN hotspot=1    : {n_hs_cnn} / {n_scored}  ({n_hs_cnn/n_scored*100:.1f}%)')
print(f'  S5P hotspot=1    : {n_hs_s5p} points with CH₄ data')
print(f'  Both agree (HS)  : {n_both} points')
print(f'  CNN mean prob    : {scored["CNN_probability"].mean():.4f}')
print(f'  CNN max prob     : {scored["CNN_probability"].max():.4f}')
print('─' * 70)
print('  SCIENTIFIC CAVEATS:')
print('  1. CNN probability reflects spectral surface conditions, not CH₄ directly.')
print('  2. S5P TROPOMI footprint ~7×5.5 km — cannot locate emission sources precisely.')
print('  3. No independent ground-truth labels available → accuracy/F1 cannot be computed.')
print('  4. GeoTIFF is IDW interpolation of sparse point predictions — not a sensor product.')
print('  5. S2 and S5P acquisitions may differ by up to ±10 days from target date.')
print('═' * 70)

if len(hotspot_df) > 0:
    pk = hotspot_df.iloc[0]
    print(f'\n  Peak CNN hotspot:')
    print(f'    Lat        : {pk["lat"]:.6f}°N')
    print(f'    Lon        : {pk["lon"]:.6f}°E')
    print(f'    CNN prob   : {pk["CNN_probability"]:.6f}')
    if pd.notna(pk.get('S5P_CH4_ppb')):
        print(f'    S5P CH4    : {pk["S5P_CH4_ppb"]:.3f} ppb  (S5P date: {pk["s5p_acquisition_date"]})')
        print(f'    Anomaly    : {pk["ch4_anomaly_ppb"]:+.3f} ppb above background')

# ── 3-panel figure ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0d1117')
fig.suptitle(f'Vellakkal — CNN Hotspot Probability  |  target {TARGET_DATE}',
             color='white', fontsize=13, fontweight='bold')

for ax in axes:
    ax.set_facecolor('#161b22')
    for sp in ax.spines.values():
        sp.set_edgecolor('#30363d')
    ax.tick_params(colors='white')

# Panel 1: spatial map
sc = axes[0].scatter(scored['lon'], scored['lat'],
                     c=scored['CNN_probability'], cmap='RdYlGn_r',
                     s=40, alpha=0.85, vmin=0, vmax=1, edgecolors='none')
axes[0].scatter([SITE_LON], [SITE_LAT], c='white', s=150, marker='*',
                zorder=10, label='Dumpsite')
cb = plt.colorbar(sc, ax=axes[0])
cb.set_label('CNN Probability', color='white')
cb.ax.yaxis.set_tick_params(color='white')
plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')
axes[0].set_title('CNN Hotspot Probability Map', color='white', fontsize=11)
axes[0].set_xlabel('Longitude', color='white')
axes[0].set_ylabel('Latitude',  color='white')
axes[0].legend(facecolor='#161b22', labelcolor='white', fontsize=9)

# Panel 2: probability distribution
axes[1].hist(scored['CNN_probability'], bins=30,
             color='#1f77b4', edgecolor='#30363d', alpha=0.85)
axes[1].axvline(0.5, color='#ff4444', lw=2, ls='--', label='Threshold 0.5')
axes[1].set_title('Probability Distribution', color='white', fontsize=11)
axes[1].set_xlabel('CNN Probability', color='white')
axes[1].set_ylabel('Count', color='white')
axes[1].legend(facecolor='#161b22', labelcolor='white', fontsize=9)

# Panel 3: CNN prob vs S5P CH4 anomaly scatter
both = scored.dropna(subset=['ch4_anomaly_ppb', 'CNN_probability'])
if len(both) > 0:
    c = ['#ff4444' if p >= 0.5 else '#44aa44' for p in both['CNN_probability']]
    axes[2].scatter(both['ch4_anomaly_ppb'], both['CNN_probability'],
                    c=c, alpha=0.7, s=30, edgecolors='none')
    axes[2].axhline(0.5, color='#aaa', lw=1, ls='--', label='CNN threshold')
    axes[2].axvline(BG_STD * ENHANCEMENT_SIGMA_THRESHOLD, color='#ff8c00',
                    lw=1.5, ls='--', label=f'S5P threshold (+{BG_STD*ENHANCEMENT_SIGMA_THRESHOLD:.1f} ppb)')
    axes[2].legend(facecolor='#161b22', labelcolor='white', fontsize=9)
    axes[2].set_title('CH₄ Anomaly vs CNN Prob', color='white', fontsize=11)
    axes[2].set_xlabel('CH₄ anomaly (ppb)', color='white')
    axes[2].set_ylabel('CNN probability', color='white')
else:
    axes[2].text(0.5, 0.5, 'Insufficient S5P\ndata for comparison',
                 ha='center', va='center', color='#8b949e',
                 fontsize=12, transform=axes[2].transAxes)
    axes[2].set_title('CH₄ Anomaly vs CNN Prob', color='white', fontsize=11)

plt.tight_layout()
fig_path = f'{OUT_DIR_FIG}/vellakkal_report_{TARGET_DATE}.png'
plt.savefig(fig_path, dpi=180, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'\n✅ Figure saved: {fig_path}')

## Step 12: Copy Outputs to Google Drive

In [ ]:
import shutil

drive_outputs = os.path.join(PROJECT_DRIVE_PATH, 'outputs')

file_map = [
    (OUT_CSV_ALL,     'predictions'),
    (OUT_CSV_HOTSPOT, 'predictions'),
    (OUT_GEOJSON,     'gis'),
    (OUT_TIF,         'gis'),
    (OUT_MAP,         'gis'),
    (fig_path,        'figures'),
]

print('Copying outputs to Google Drive...')
for src, subdir in file_map:
    dst_dir = os.path.join(drive_outputs, subdir)
    os.makedirs(dst_dir, exist_ok=True)
    if os.path.exists(src):
        shutil.copy2(src, dst_dir)
        size_kb = os.path.getsize(src) / 1024
        print(f'  ✅ {os.path.basename(src):55s} ({size_kb:.0f} KB)')
    else:
        print(f'  ⚠️  Missing: {src}')

print(f'\n🎉 Done. All outputs in Drive: {drive_outputs}')
print(f'\nRequired file checklist:')
required = [
    OUT_CSV_HOTSPOT,
    OUT_GEOJSON,
    OUT_TIF,
    OUT_MAP,
]
for f in required:
    exists = os.path.exists(f) or os.path.exists(
        os.path.join(drive_outputs, os.path.dirname(f).split('/')[-1], os.path.basename(f)))
    status = '✅' if os.path.exists(f) else '❌'
    print(f'  {status} {os.path.basename(f)}')